# Master — KIČMENA MOŽDINA (nnU-Net)

Model za lezije u kičmenoj moždini na MS-Multi-Spine skupu.


## Ćelija 0 — priprema

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q nnunetv2 nibabel

import os, shutil, json, glob, re
import numpy as np
import nibabel as nib

MASTER = '/content/drive/MyDrive/Master/kicma'
os.environ['nnUNet_raw']          = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results']      = f'{MASTER}/modeli/nnUNet_results'
for p in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
    os.makedirs(p, exist_ok=True)

import torch
print('GPU:', torch.cuda.is_available(), '(za pripremu nije nužan)')
print('Spremno.')

## Ćelija 1 — raspakivanje + pregled strukture

In [ ]:
import tarfile
tar_path = f'{MASTER}/podaci_sirovi/MS_MultiSpine_dataset.tar.gz'
print('tar postoji:', os.path.exists(tar_path),
      '| GB:', round(os.path.getsize(tar_path)/1e9,2) if os.path.exists(tar_path) else '-')

if not os.path.exists('/content/multispine'):
    print('raspakivam (nekoliko min)...')
    with tarfile.open(tar_path) as t:
        t.extractall('/content/multispine')
    print('gotovo.')

base = '/content/multispine'

print('\n--- STABLO ---')
for root, dirs, files in sorted(os.walk(base)):
    dirs[:] = [d for d in dirs if not d.startswith('__')]
    rel = os.path.relpath(root, base)
    depth = 0 if rel=='.' else rel.count(os.sep)+1
    if depth > 3:
        dirs[:] = []; continue
    ime = 'ROOT' if rel=='.' else os.path.basename(root)
    print('   '*depth + ime + '/   ' + (str(sorted(files)[:8]) if files else ''))

svi = glob.glob(f'{base}/**/*.nii*', recursive=True)
def je(f, *k): n=os.path.basename(f).lower(); return any(x in n for x in k)
maske = [f for f in svi if je(f,'lesion','_les','mask','seg','_gt','label')]
print('\n--- KLASIFIKACIJA ---')
print('ukupno .nii:', len(svi), '| kandidata maski:', len(maske))
for seq in ['t2','stir','psir','mp2rage','t1']:
    n = [f for f in svi if je(f,seq) and f not in maske]
    print(f'  {seq.upper()}: {len(n)}   primer: {os.path.basename(n[0]) if n else "-"}')
print('  primer maske:', os.path.basename(maske[0]) if maske else '-')

## Ćelija 2 — pravljenje nnU-Net formata (Dataset011_Spine)


In [ ]:
def je(f, *k): n=os.path.basename(f).lower(); return any(x in n for x in k)
svi = glob.glob('/content/multispine/**/*.nii*', recursive=True)
maske = [f for f in svi if je(f,'lesion','_les','mask','seg','_gt','label')]
t2    = [f for f in svi if je(f,'t2') and f not in maske]

def zaj(a, b):
    a,b = a.split('/'), b.split('/'); n=0
    for x,y in zip(a,b):
        if x==y: n+=1
        else: break
    return n

parovi = []
for t in t2:
    if not maske: break
    m = max(maske, key=lambda x: zaj(t,x))
    if zaj(t,m) >= len(t.split('/'))-2:
        parovi.append((t,m))
print('T2 fajlova:', len(t2), '| maski:', len(maske), '| uparenih:', len(parovi))
if parovi:
    print('primer para:\n  T2  :', parovi[0][0].replace("/content/multispine/",""),
          '\n  maska:', parovi[0][1].replace("/content/multispine/",""))

assert parovi, 'Nema uparenih! Pošalji Claudeu ispis iz Ćelije 1 pa ćemo popraviti nazive.'

DS = os.environ['nnUNet_raw'] + '/Dataset011_Spine'
for sub in ['imagesTr','labelsTr']:
    os.makedirs(f'{DS}/{sub}', exist_ok=True)

def upisi_masku(src, dst):
    img = nib.load(src)
    d = (np.asarray(img.dataobj) > 0).astype(np.uint8)
    nib.save(nib.Nifti1Image(d, img.affine, img.header), dst)

n = 0
for i,(t,m) in enumerate(parovi):
    cid = f'spine_{i:03d}'
    try:
        shutil.copy(t, f'{DS}/imagesTr/{cid}_0000.nii.gz')
        upisi_masku(m, f'{DS}/labelsTr/{cid}.nii.gz')
        n += 1
    except Exception as e:
        print('GREŠKA', cid, e)

with open(f'{DS}/dataset.json','w') as f:
    json.dump({"channel_names":{"0":"T2"},
               "labels":{"background":0,"lesion":1},
               "numTraining":n,"file_ending":".nii.gz",
               "overwrite_image_reader_writer":"SimpleITKIO"}, f, indent=2)
print('upisano slučajeva:', n, '| dataset.json OK')

## Ćelija 3 — nnU-Net plan & preprocess

In [ ]:
!nnUNetv2_plan_and_preprocess -d 11 --verify_dataset_integrity

## Ćelija 4 — TRENING (GPU)


In [ ]:
rez = f"{os.environ['nnUNet_results']}/Dataset011_Spine/nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres/fold_0"
nastavi = "--c" if os.path.exists(f"{rez}/checkpoint_latest.pth") else ""
print("Nastavljam od checkpointa." if nastavi else "Krećem od nule.")
!nnUNetv2_train 11 3d_fullres 0 -tr nnUNetTrainer_100epochs {nastavi}